# Orcrist, area A (metrics): training and tuning, step by step

The workbench to train Orcrist and choose its parameters, one version at a time. It is the companion of `train-pca-B0.ipynb`, with the same cross-validation (CV, in `anomaly/models/cv.py`) and the same protocol (`anomaly/models/evaluate.py`), plus what only a network has: the training curves, the seeds, and long searches in background. Run it once from the top, then change the cells marked **edit** and rerun from there.

Steps:
1. **Folds**: the clean train and val sessions in K folds of whole sessions (the same folds as the PCA).
2. **One CV by hand**: set every parameter and train one model per fold, watching each epoch; then the training curves, what the model learned, who raises false alarms and why, the separation, each dev fault.
3. **Seeds**: the same CV with other seeds. Their spread is the noise floor: differences between configurations smaller than it are not real.
4. **Sweep**: one parameter over a list of values, the others fixed, to choose where the search looks.
5. **Optuna**: the search space (a choice or a range for any parameter), the sampler, the budget, in this notebook or in background processes.
6. **The study**: the trials, the Pareto front of F1 and top@3, which parameters matter, any trial opened as in step 2.
7. **Choice**: the best candidates of the rule confirmed with several seeds, against the defaults, and the record in `results/tuning/orcrist-<version>-B0/<app>/`.

How the CV works (as for the PCA):
- **fold i**: Orcrist is trained on the other folds and stopped early on fold i (the epoch where fold i looks most normal: a slight optimism, the same for every configuration); the threshold of each service is its largest persistent score over the fit sessions; then the model scores fold i (healthy: every alarm there is false) and every dev fault;
- **pooled**: every inference counts once: false alarms summed over the folds, fault inferences averaged over the K models, and so found, minutes and top@k;
- **objectives**: F1 and top@3 of the reference system (`model+restart+absence`).

The faults choose the parameters, never the weights. Test, test-unseen, noop and the test faults are never read here. Fixed, as the protocol shared with the PCA: the warm-up, the persistence (2 steps), the threshold, the rules and the variants.

Cost: one fit takes 15-40 s (more with larger H, K or LAYERS), so one CV is 5 fits, about 2-3 minutes; a search of 40 trials is 1.5-2 hours in this notebook, or about 30 minutes with 4 background workers. At most 4 workers run on the machine (4 × 8 threads = the 32 cores): more make every fit crawl.

Terms: **degree** = score / threshold, above 1 a service is in alarm; **healthy in alarm %** = the share of held-out healthy steps in alarm; **found** = an alarm starts after the injection; **top@k** = a target is among the first k services ranked at that first alarm; **NLL** = the reconstruction term of the loss, **KL** = the latent term; **active dimensions** = latent dimensions whose KL on val is above 0.01.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from anomaly import build
from anomaly.models import cv, evaluate as ev, orcrist
from anomaly.models.orcrist import model as om

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 3)

APP = "hotel"  # edit: "hotel" or "social"
VERSION = "AB"  # edit: a key of orcrist.VERSIONS (AA, AB, AC, AD)
K_FOLDS = 5  # edit: folds of the CV
EVERY = 25  # edit: while training, a line every EVERY epochs (0: none)
DATA = build.DATASET / "built"
OUT = cv.OUT  # studies and choices go to OUT/orcrist-<version>-B0/<app>/
V = cv.VARIANT  # the variant the figures follow: the reference system
NAME = f"orcrist-{VERSION}"

RUNS = ev.load(APP, DATA)
SERVICES = RUNS[0].services
FEATURES = list(om.Orcrist.features)
print(f"{APP}, {NAME} {orcrist.VERSIONS[VERSION]}: {len(RUNS)} runs, {len(SERVICES)} services; "
      f"{sum(r.role in ('train', 'val') for r in RUNS)} clean train and val sessions, "
      f"{sum(r.role == 'fault-dev' for r in RUNS)} dev faults")

In [ ]:
# Colours: categorical slots in a fixed order (as in pca-B0 and orcrist-A); sequential blue for good, orange for bad
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, YELLOW, RED, NEUTRAL, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#f0efec", "#b9b7b1"
PURPLE = "#7f56d9"
FOLD_COLOUR = [BLUE, ORANGE, AQUA, YELLOW, PURPLE, INK2, MUTED]
DIVERGING = LinearSegmentedColormap.from_list("ratio", [BLUE, NEUTRAL, RED])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f2f7fd", "#cde2fb", "#2a78d6", "#104281"])
SEQUENTIAL_BAD = LinearSegmentedColormap.from_list("bad", ["#fdf4ef", "#f9cdb8", "#eb6834", "#8a3110"])  # darker = worse
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False, "lines.linewidth": 2,
    "figure.dpi": 110})


def rep(r):
    """The repetition of a run: r13."""
    return r.id.split("__")[-1]


def batch(r):
    return "r1-r12" if int(rep(r)[1:]) <= 12 else "r13+"


def heat(ax, M, cmap, labels, fmt, vmin=None, vmax=None, dark=.6):
    """A heatmap with every cell written; NaN cells left blank."""
    ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    lo, hi = np.nanmin(M) if vmin is None else vmin, np.nanmax(M) if vmax is None else vmax
    for (i, j), x in np.ndenumerate(M):
        if np.isfinite(x):
            ax.text(j, i, fmt(x), ha="center", va="center", fontsize=8,
                    color="white" if hi > lo and (x - lo) / (hi - lo) > dark else INK)
    ax.set_yticks(range(len(labels)), labels)
    ax.grid(False)


def watch(fold, row):
    """Called at every epoch of every fold while training: one line every EVERY epochs."""
    if EVERY and (row["epoch"] == 1 or row["epoch"] % EVERY == 0):
        print(f"fold {fold}, epoch {row['epoch']:>3}: KL weight {row['beta']:.2f}; train {row['train_loss']:.3f} "
              f"(NLL {row['train_nll']:.3f}, KL {row['train_kl']:.3f}); val {row['val_loss']:.3f} "
              f"(NLL {row['val_nll']:.3f}, KL {row['val_kl']:.3f}); {row['active_dims']} active dims; "
              f"{row['seconds']:.0f} s", flush=True)

## 1. Folds

The clean train and val sessions, whole, in K folds balanced by loaded steps (the same as in `train-pca-B0.ipynb`). Each fold is held out once: the fit uses the other K − 1, about (K − 1)/K of the steps the final model trains on.

In [ ]:
FOLDS = cv.folds(RUNS, K_FOLDS)
steps = lambda f: sum(int(r.on.sum()) for r in f)
display(pd.DataFrame([{"fold": i, "sessions": " ".join(rep(r) + (" (val)" if r.role == "val" else "") for r in f),
                       "steps": steps(f), "hours": round(steps(f) * 30 / 3600, 1),
                       "r1-r12": sum(batch(r) == "r1-r12" for r in f), "r13+": sum(batch(r) == "r13+" for r in f)}
                      for i, f in enumerate(FOLDS)]).set_index("fold"))

## 2. One cross-validation by hand

**edit** `HP`: every parameter of Orcrist (the defaults are `om.HP`, those of `results/orcrist-<version>-B0`):
- architecture: `D` the size of the service's condition (embedding; AD uses a one-hot instead), `H` the hidden units per layer, `LAYERS` the hidden layers of the encoder and of the decoder, `K` the latent size;
- likelihood and latent: `SIGMA_MIN` the floor of the decoder's deviation in log1p units (the narrowest normal range: lower is more sensitive, and more false alarms on near-flat features), `BETA` the weight of the KL once warmed up (1 is the ELBO; below 1 the reconstruction is sharper and the latent less regular), `KL_WARMUP` the epochs for that weight to grow from 0, `INSTANCE_DROPOUT` (AB, AC) the probability of dropping the instance embedding in training, which pulls a service towards its type;
- optimisation: `LR` and `WEIGHT_DECAY` (Adam), `BATCH`, `MAX_EPOCHS`, `PATIENCE` (epochs without a better val loss before stopping, counted after the warm-up);
- cells and seed: `MIN_FINITE`, `MIN_SD` (which cells enter the model, as in the PCA), `SEED` (initialisation, batches, sampling, dropout).

The next cell defines the tables and figures of this section; section 6 reuses them to open any trial.

In [ ]:
def tables(res):
    """Per fold (the reference system) and pooled (every variant)."""
    display(res.table[res.table.variant == V].drop(columns="variant").set_index("fold").round(3))
    display(res.pooled.T.round(3))


def training_table(res):
    """Per fold: how long it trained and why it stopped, and the gap between train and val at the best epoch (both in
    evaluation mode, with the final KL weight)."""
    rows = []
    for f in res.folds:
        p = f.model.params
        rows.append({"fold": f.i, "epochs": p["epochs"], "best epoch": p["best_epoch"],
                     "stopped by": "MAX_EPOCHS" if p["epochs"] == p["MAX_EPOCHS"] else "PATIENCE",
                     "train loss": p["train_loss"], "val loss": p["val_loss"], "gap": p["val_loss"] - p["train_loss"],
                     "active dims": p["active_dims"], "parameters": p["parameters"], "seconds": p["seconds"]})
    return pd.DataFrame(rows).set_index("fold")


def fig_training(res):
    """The curves of every fold: dashed train (as trained: sampled z, dropout, the KL weight of the epoch), solid val
    (the held-out fold, with the final KL weight); the dot is the best epoch, the grey band the KL warm-up."""
    hist = [pd.DataFrame(f.model.history) for f in res.folds]
    warm = res.folds[0].model.params["KL_WARMUP"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 7.4), sharex=True)
    panels = (("loss", "train_loss", "val_loss"), ("NLL (reconstruction)", "train_nll", "val_nll"),
              ("KL (latent)", "train_kl", "val_kl"), ("active latent dimensions on val", None, "active_dims"))
    for ax, (title, tr, va) in zip(axes.flat, panels):
        shown = []
        for f, h in zip(res.folds, hist):
            c = FOLD_COLOUR[f.i % len(FOLD_COLOUR)]
            if tr:
                ax.plot(h.epoch, h[tr], color=c, lw=1, ls="--")
                shown.append(h[tr][h.epoch > 3])
            ax.plot(h.epoch, h[va], color=c, lw=1.6)
            shown.append(h[va][h.epoch > 3])
            b = f.model.params["best_epoch"]
            ax.scatter([b], [h[va].iloc[b - 1]], color=c, s=28, zorder=3)
        ax.axvspan(0, warm, color=NEUTRAL, zorder=0)
        v = pd.concat(shown)
        if len(v) and tr:
            pad = .05 * (v.max() - v.min() or 1)
            ax.set_ylim(v.min() - pad, v.max() + pad)
        ax.set_title(title)
    for ax in axes[1]:
        ax.set_xlabel("epoch")
    fig.legend(handles=[Line2D([], [], color=FOLD_COLOUR[i % len(FOLD_COLOUR)], label=f"fold {i}") for i in range(K_FOLDS)]
               + [Line2D([], [], color=INK2, ls="--", lw=1, label="train"), Line2D([], [], color=INK2, label="val"),
                  Line2D([], [], ls="", marker="o", color=INK2, label="best epoch"),
                  Patch(color=NEUTRAL, label="KL warm-up")], loc="upper center", ncol=K_FOLDS + 4)
    fig.tight_layout(rect=(0, 0, 1, .94))
    plt.show()


def fig_model(res, fold=0):
    """What Orcrist learned: where the decoder deviation sits at its floor on the held-out steps and the KL per latent
    dimension (one fold), and how the threshold of each service moves between folds."""
    f = res.folds[fold]
    floor = f.model.params["SIGMA_MIN"] * 1.02
    on = [r for r in f.held]
    sd = np.concatenate([r.sd[r.on] for r in on])
    at_floor = np.where(f.model.cols, (sd <= floor).mean(0), np.nan)
    kl = f.model.kl_dims
    thr = np.array([x.thr for x in res.folds]).T
    ratio = thr / np.nanmedian(np.where(np.isfinite(thr), thr, np.nan), 1, keepdims=True)
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, .32 * len(SERVICES) + 2.4), sharey=True,
                                     gridspec_kw={"width_ratios": [len(FEATURES), max(kl.shape[1], 2), K_FOLDS]})
    heat(a1, 100 * at_floor, SEQUENTIAL_BAD, SERVICES, lambda x: f"{x:.0f}", 0, 100)
    a1.set_xticks(range(len(FEATURES)), FEATURES, rotation=40, ha="right")
    a1.set_title(f"held-out steps with the deviation at its floor %\n(fold {fold}; blank: cell out of the model)")
    heat(a2, kl, SEQUENTIAL, SERVICES, lambda x: f"{x:.2f}")
    a2.set_xticks(range(kl.shape[1]), [f"z{d + 1}" for d in range(kl.shape[1])])
    a2.set_title(f"mean KL per latent\ndimension, fold {fold}")
    heat(a3, np.log2(ratio), DIVERGING, SERVICES, lambda x: f"{2 ** x:.2f}", -1, 1, dark=2)
    a3.set_xticks(range(K_FOLDS))
    a3.set(xlabel="fold", title="threshold / its median\nover the folds")
    fig.tight_layout()
    plt.show()


def false_alarms(res):
    """Per held-out healthy step and service in alarm: the fold, the service, the features that make the score (share)."""
    rows = []
    for f in res.folds:
        for r in f.held:
            for t, i in zip(*np.where(r.alarm[V] & r.on[:, None])):
                c = r.r2[t, i]
                rows.append({"fold": f.i, "session": rep(r), "service": SERVICES[i], "degree": r.norm[t, i],
                             "rule": bool(r.rule[V][t, i]), **dict(zip(FEATURES, c / c.sum() if c.sum() else c))})
    return pd.DataFrame(rows, columns=["fold", "session", "service", "degree", "rule", *FEATURES])


def fig_false_alarms(res):
    fa = false_alarms(res)
    count = fa.groupby(["service", "fold"]).size().unstack().reindex(index=SERVICES, columns=range(K_FOLDS)).fillna(0)
    share = fa.groupby("service")[FEATURES].mean().reindex(SERVICES)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, .32 * len(SERVICES) + 2.4), sharey=True,
                                 gridspec_kw={"width_ratios": [K_FOLDS + 1, len(FEATURES)]})
    heat(a1, np.c_[count.values, count.sum(axis=1).values], SEQUENTIAL_BAD, SERVICES, lambda x: f"{x:.0f}" if x else "")
    a1.set_xticks(range(K_FOLDS + 1), [*range(K_FOLDS), "all"])
    a1.set(xlabel="fold", title="held-out healthy steps in alarm")
    heat(a2, 100 * share.values, SEQUENTIAL_BAD, SERVICES, lambda x: f"{x:.0f}", 0, 100)
    a2.set_xticks(range(len(FEATURES)), FEATURES, rotation=40, ha="right")
    a2.set_title("what makes those alarms: mean share of the score %")
    fig.suptitle(f"false alarms: {len(fa)} service-steps, {fa.rule.sum()} by a rule", fontweight="bold")
    fig.tight_layout()
    plt.show()


def fig_separation(res):
    """ECDF of the system degree (the largest over the services) on held-out healthy steps and on dev fault steps."""
    h = np.concatenate([r.norm[r.on].max(1) for f in res.folds for r in f.held])
    x = np.concatenate([r.norm[r.fault].max(1) for f in res.folds for r in f.dev])
    fig, ax = plt.subplots(figsize=(9, 4))
    for d, c, name in ((h, BLUE, "held-out healthy steps"), (x, ORANGE, "dev fault steps")):
        d = np.sort(np.clip(d, 1e-2, None))
        ax.step(d, np.arange(1, len(d) + 1) / len(d), where="post", color=c,
                label=f"{name}: {100 * (d > 1).mean():.1f}% above 1")
    ax.axvline(1, color=INK2, ls=":", lw=1)
    ax.set_xscale("log")
    ax.set(xlabel="system degree (largest score / threshold over the services; model only, no rule)",
           ylabel="share of steps at or below", title="separation of healthy and faulty steps, all folds")
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()


def first_rank(r):
    """Rank of the best ranked target at the first alarm after the injection (1 = first), NaN when not found."""
    first = ev.starts(r.alarm[V]).any(1) & r.fault
    if not first.any():
        return np.nan
    k = int(first.argmax())
    order = list(ev.rank(r.norm[k], r.rule[V][k]))
    return min(order.index(t) for t in r.targets) + 1


def fig_faults(res):
    """Each dev fault in each fold: the rank of its target at the first alarm; blank = not found."""
    dev = [r for r in res.folds[0].dev if r.targets]
    R = np.array([[first_rank(r) for r in f.dev if r.targets] for f in res.folds]).T
    fig, ax = plt.subplots(figsize=(7, .27 * len(dev) + 1.8))
    heat(ax, np.minimum(R, 6), SEQUENTIAL.reversed(), [f"{r.scenario} {rep(r)}" for r in dev],
         lambda x: "", 1, 6)
    for (i, j), x in np.ndenumerate(R):
        ax.text(j, i, "miss" if np.isnan(x) else f"{x:.0f}", ha="center", va="center", fontsize=8,
                color="white" if x <= 2 else INK)
    ax.set_xticks(range(K_FOLDS))
    ax.set(xlabel="fold", title="rank of the target at the first alarm\n(1-3: top@3; miss: not found)")
    fig.tight_layout()
    plt.show()


def show_all(res):
    tables(res)
    display(training_table(res).round(3))
    fig_training(res)
    fig_model(res)
    fig_false_alarms(res)
    fig_separation(res)
    fig_faults(res)

In [ ]:
HP = {  # edit
    # architecture
    "D": 4,
    "H": 32,
    "LAYERS": 2,
    "K": 4,
    # likelihood and latent
    "SIGMA_MIN": 0.1,
    "BETA": 1.0,
    "KL_WARMUP": 20,
    "INSTANCE_DROPOUT": 0.2,
    # optimisation
    "LR": 1e-3,
    "WEIGHT_DECAY": 0.0,
    "BATCH": 512,
    "MAX_EPOCHS": 300,
    "PATIENCE": 20,
    # cells and seed
    "MIN_FINITE": 0.95,
    "MIN_SD": 1e-6,
    "SEED": 0,
}
assert set(HP) == set(om.HP), f"HP must list every parameter: {set(HP) ^ set(om.HP)}"
res = cv.cross_validate(NAME, RUNS, HP, K_FOLDS, on_epoch=watch)
tables(res)

### 2a. Training

How long each fold trained, why it stopped, and the gap between the training and the held-out loss at the best epoch (in evaluation mode: a large gap is overfitting to the fit sessions). Then the curves: a val loss that keeps falling at `MAX_EPOCHS` asks for more epochs; one that turns up early, for a lower `LR` or more regularisation; active dimensions that fall to 1 show the latent collapsing to one factor (the load), often fine for metrics.

In [ ]:
display(training_table(res).round(3))
fig_training(res)

### 2b. What the model learned, and how stable it is

Left: where the decoder's deviation sits at its floor on the held-out steps. A cell at its floor 100% of the time has a normal range set by `SIGMA_MIN`, not learned: lowering the floor makes it more sensitive. Middle: the KL of each latent dimension per service, the dimensions the model uses. Right: the threshold of each service in each fold over its median: one that moves a lot depends on which sessions were fit.

In [ ]:
fig_model(res)

### 2c. False alarms: who and why

Every held-out healthy step where a service is in alarm, per service and fold, and the features that make their score.

In [ ]:
fig_false_alarms(res)
false_alarms(res).sort_values("degree", ascending=False).head(15).round(2)

### 2d. Separation and each fault

The system degree on held-out healthy steps against dev fault steps (the threshold is the line at 1), then every dev fault in every fold.

In [ ]:
fig_separation(res)
fig_faults(res)

## 3. Seeds: the noise floor

**edit** `SEEDS`: the same CV with each seed (the other parameters as in `HP`). The deviation across seeds is how much a result moves by chance alone: a difference between two configurations smaller than it, or than one dev fault (about 4 points of top@3 or found), is noise. One CV per seed (2-3 minutes each).

In [ ]:
SEEDS = [0, 1, 2]  # edit
KEY = ["F1", "precision", "recall", "healthy in alarm %", "faults found %", "top@1 %", "top@3 %", "top@5 %",
       "minutes to alarm, median"]
seeds = pd.DataFrame([{"SEED": s, **cv.cross_validate(NAME, RUNS, HP | {"SEED": s}, K_FOLDS).pooled.loc[V, KEY]}
                      for s in SEEDS]).set_index("SEED")
display(seeds.round(3))
seeds.agg(["mean", "std"]).round(3)

## 4. Sweep: one parameter at a time

**edit** `SWEEP`: a parameter of `HP` and the values to try; the others stay as in `HP`. One CV per value. The thick line is pooled; the dots are the folds (F1 is pooled only); the dashed line is the value in `HP`. Where the curves are flat the parameter matters little; where they bend is the region to search. Compare the steps with the seed deviation of section 3.

In [ ]:
SWEEP = ("SIGMA_MIN", [0.02, 0.05, 0.1, 0.2, 0.3])  # edit
param, values = SWEEP
rows, fold_rows = [], []
for x in values:
    r_ = cv.cross_validate(NAME, RUNS, HP | {param: x}, K_FOLDS)
    rows.append({param: x, **r_.pooled.loc[V].to_dict()})
    fold_rows += [{param: x, **row} for row in r_.table[r_.table.variant == V].to_dict("records")]
    print(f"{param} = {x}: F1 {rows[-1]['F1']:.3f}, top@3 {rows[-1]['top@3 %']:.0f}%", flush=True)
sweep, sweep_folds = pd.DataFrame(rows), pd.DataFrame(fold_rows)

METRICS = ["F1", "top@3 %", "healthy in alarm %", "recall", "faults found %", "top@1 %"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
log = min(values) > 0 and max(values) / min(values) >= 20
for ax, m in zip(axes.flat, METRICS):
    if m in sweep_folds:
        ax.scatter(sweep_folds[param], sweep_folds[m], s=14, color=MUTED, zorder=2)
    ax.plot(sweep[param], sweep[m], color=BLUE, marker="o", ms=5, zorder=3)
    ax.axvline(HP[param], color=INK2, ls="--", lw=1)
    ax.set_title(m)
    if log:
        ax.set_xscale("log")
for ax in axes[1]:
    ax.set_xlabel(param)
fig.suptitle(f"{APP}, {NAME}: sweep of {param} (the rest as in HP)", fontweight="bold")
fig.tight_layout()
plt.show()
sweep.set_index(param)[KEY + ["healthy in alarm % sd", "top@3 % sd"]].round(3)

## 5. Optuna

**edit** `SPACE`: every parameter of Orcrist, each one
- a list, a choice among values: `[16, 32, 64]`;
- `(lo, hi)`, a range (of integers when both are integers): `(0.0, 0.5)`;
- `(lo, hi, "log")`, a range in log scale: `(3e-4, 3e-3, "log")`;
- any other value, fixed: `512`.

The parameters this version does not use are fixed below the space (AD has no embedding size; only AB and AC drop the instance).

`SAMPLER`: `"tpe"` (Bayesian, multi-objective), `"random"`, or `"grid"` (every combination of the lists, no range). The study stops when it has `N_TRIALS` complete trials, after `TIMEOUT_H` hours, or when a grid is done; a tpe or random study starts with the defaults (the reference) when they are in the space.

`STUDY`: kept in `OUT/orcrist-<version>-B0/<app>/study.journal` (not in git); rerunning resumes it. A study refuses another space, sampler, folds or dev set: a new space is a new name.

`BACKGROUND = True` runs the study in `WORKERS` processes detached from this notebook (they go on if the kernel stops, and pause during the nightly collection); the next cell follows them, section 6 reads the trials done so far. `False` runs it here, a line per trial.

In [ ]:
SPACE = {  # edit
    "D": [2, 4, 8],
    "H": [16, 32, 64],
    "LAYERS": [1, 2, 3],
    "K": [2, 4, 8],
    "SIGMA_MIN": (0.02, 0.3, "log"),
    "BETA": (0.25, 1.0),
    "KL_WARMUP": 20,
    "INSTANCE_DROPOUT": (0.0, 0.5),
    "LR": (3e-4, 3e-3, "log"),
    "WEIGHT_DECAY": 0.0,
    "BATCH": 512,
    "MAX_EPOCHS": 300,
    "PATIENCE": 20,
    "MIN_FINITE": 0.95,
    "MIN_SD": 1e-6,
    "SEED": 0,
}
if orcrist.VERSIONS[VERSION]["condition"] == "onehot":
    SPACE["D"] = HP["D"]  # AD: a one-hot, no embedding size
if orcrist.VERSIONS[VERSION]["condition"] != "type+instance":
    SPACE["INSTANCE_DROPOUT"] = HP["INSTANCE_DROPOUT"]  # only AB and AC drop the instance
assert set(SPACE) == set(om.HP), f"SPACE must list every parameter: {set(SPACE) ^ set(om.HP)}"
SAMPLER = "tpe"  # edit: "tpe", "random" or "grid"
STUDY = f"{NAME}-{APP}-1"  # edit: a new name for a new space
N_TRIALS = 40  # edit
TIMEOUT_H = None  # edit: hours
BACKGROUND = True  # edit
WORKERS = 4  # edit: at most cv.MAX_WORKERS on the machine, all studies together

study = cv.study(NAME, APP, RUNS, SPACE, STUDY, SAMPLER, K_FOLDS, out=OUT)
if BACKGROUND:
    cv.launch(study, NAME, APP, SPACE, SAMPLER, N_TRIALS, TIMEOUT_H, K_FOLDS, WORKERS, DATA, OUT)
else:
    cv.optimize(study, NAME, RUNS, SPACE, N_TRIALS, TIMEOUT_H, K_FOLDS)

In [ ]:
# Rerun to follow the background workers; to stop them: cv.stop(study, NAME, APP, OUT)
if BACKGROUND:
    cv.workers(study, NAME, APP, OUT)

## 6. The study

The complete trials so far, best F1 first (rerun while the workers run to refresh). `pareto`: no other trial is better on both F1 and top@3. `reference`: the default parameters. The `sd` columns are the deviation across folds; the seed deviation of section 3 is the other measure of noise.

In [ ]:
df = cv.trials(study)
PARAMS = [p for p, s in SPACE.items() if isinstance(s, (list, tuple))]
COLS = PARAMS + ["F1", "precision", "recall", "healthy in alarm %", "faults found %", "top@1 %", "top@3 %", "top@5 %",
                 "minutes to alarm, median", "healthy in alarm % sd", "top@3 % sd", "pareto", "reference"]
print(f"{len(df)} complete trials")
df[COLS].sort_values("F1", ascending=False).round(3).head(25)

In [ ]:
chosen = cv.choose(df)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.2, 1, 1]})
a1.scatter(df.F1, df["top@3 %"], s=22, color=MUTED, label="trial")
front = df[df.pareto].sort_values("F1")
a1.plot(front.F1, front["top@3 %"], color=BLUE, marker="o", ms=6, lw=1.2, label="Pareto front")
ref = df[df.reference]
a1.scatter(ref.F1, ref["top@3 %"], s=110, marker="*", color=ORANGE, zorder=4, label="reference (defaults)")
a1.scatter([chosen.F1], [chosen["top@3 %"]], s=140, facecolor="none", edgecolor=INK, lw=1.6, zorder=5,
           label=f"rule, one seed (trial {chosen.name})")
a1.set(xlabel="F1", ylabel="top@3 %", title="F1 against top@3")
a1.legend(loc="upper left", fontsize=8)
for ax, m in ((a2, "F1"), (a3, "top@3 %")):
    ax.scatter(df.index, df[m], s=14, color=MUTED)
    ax.plot(df.index, df[m].cummax(), color=BLUE, lw=1.5, label="best so far")
    ax.set(xlabel="trial", title=f"{m} by trial")
    ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

Which parameters matter (PED-ANOVA: how much each parameter's values differ between the best trials and the rest, per objective; from 10 trials), and every trial against every parameter.

In [ ]:
from optuna.importance import PedAnovaImportanceEvaluator, get_param_importances

warnings.filterwarnings("ignore", message="PedAnovaImportanceEvaluator computes")  # handled: the target is negated
if len(df) >= 10 and len(PARAMS) >= 2:
    imp = pd.DataFrame({o: get_param_importances(study, evaluator=PedAnovaImportanceEvaluator(), params=PARAMS,
                                                 target=lambda t, j=j: -t.values[j])  # it ranks low values first
                        for j, o in enumerate(cv.OBJECTIVES)}).reindex(PARAMS)
    display(imp.round(3))
n = len(PARAMS)
cols = min(4, n)
for m in cv.OBJECTIVES:
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(3.6 * cols, 2.9 * -(-n // cols) + .7), sharey=True,
                             squeeze=False)
    for ax, p in zip(axes.flat, PARAMS):
        ax.scatter(df[p], df[m], s=18, c=np.where(df.pareto, BLUE, MUTED))
        if isinstance(SPACE[p], tuple) and SPACE[p][2:] == ("log",):
            ax.set_xscale("log")
        ax.set_xlabel(p)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    fig.legend(handles=[Line2D([], [], ls="", marker="o", color=BLUE, label="Pareto"),
                        Line2D([], [], ls="", marker="o", color=MUTED, label="other")], loc="upper right", ncol=2)
    fig.suptitle(f"{m} against every parameter", fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, .95))
    plt.show()

### 6a. Open a trial

**edit** `TRIAL`: a trial number of the table (None: the one the rule chooses on one seed). Its CV is rerun with its parameters, watching the training, and shown as in section 2.

In [ ]:
TRIAL = None  # edit
t = int(chosen.name) if TRIAL is None else TRIAL
hp = study.trials[t].user_attrs["hp"]
print(f"trial {t}: {hp}")
res_t = cv.cross_validate(NAME, RUNS, hp, K_FOLDS, on_epoch=watch)
show_all(res_t)

## 7. Choice, confirmed with seeds

One seed per trial makes the best trials partly lucky (the winner's curse). So the rule is applied twice:
1. on the trials: the Pareto front, then those whose top@3 is within `cv.TOLERANCE` points (about one dev fault) of the best, by F1: the first `TOP` are the candidates, with the reference;
2. on the candidates rerun with `CONFIRM_SEEDS`: the same rule on the means over the seeds.

A gain over the reference within the deviation across seeds is noise, and the defaults are as good. Cost: (`TOP` + 1) × seeds CVs, 2-3 minutes each.

**edit** `SAVE = True` writes `trials.csv`, `confirm.csv` and `choice.json` (chosen and reference trial, the rule, the folds, the data, the code) to `OUT/orcrist-<version>-B0/<app>/`. These are the record of the study; the final parameters are frozen only on the complete dataset, as a new version, and the test is read once, with them.

In [ ]:
TOP = 3  # edit
CONFIRM_SEEDS = [0, 1, 2, 3, 4]  # edit
front = df[df.pareto]
near = front[front["top@3 %"] >= front["top@3 %"].max() - cv.TOLERANCE].sort_values("F1", ascending=False)
candidates = list(dict.fromkeys([*near.index[:TOP], *df.index[df.reference]]))
rows = []
for t in candidates:
    hp = study.trials[t].user_attrs["hp"]
    for s in CONFIRM_SEEDS:
        rows.append({"trial": t, "SEED": s, **cv.cross_validate(NAME, RUNS, hp | {"SEED": s}, K_FOLDS).pooled.loc[V, KEY]})
        print(f"trial {t}, seed {s}: F1 {rows[-1]['F1']:.3f}, top@3 {rows[-1]['top@3 %']:.0f}%", flush=True)
confirm = pd.DataFrame(rows)
means = confirm.groupby("trial")[KEY].agg(["mean", "std"])
display(means.round(3))
m = means.xs("mean", axis=1, level=1)
best = m[m["top@3 %"] >= m["top@3 %"].max() - cv.TOLERANCE]["F1"].idxmax()
chosen = df.loc[best]
print(f"chosen on the means over {len(CONFIRM_SEEDS)} seeds: trial {best}"
      f"{' (the reference)' if chosen.reference else ''}: {study.trials[best].user_attrs['hp']}")

In [ ]:
SAVE = False  # edit
if SAVE:
    print("written to", cv.save(study, NAME, APP, chosen, OUT, confirm=confirm.set_index(["trial", "SEED"])))